In [1]:
import sys
print(sys.executable)


/venv/main/bin/python


In [6]:
import os
from pathlib import Path

ROOT = Path("/workspace/ta_finetune")
DATA = ROOT / "data"
MANIFEST = DATA / "manifest_strict.csv"
AUDIO_DIR = DATA / "preprocessed_full"

HF_CACHE = DATA / "hf_cache"
(HF_CACHE / "hub").mkdir(parents=True, exist_ok=True)
(HF_CACHE / "transformers").mkdir(parents=True, exist_ok=True)
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("MANIFEST exists:", MANIFEST.exists())
print("AUDIO_DIR exists:", AUDIO_DIR.exists())


MANIFEST exists: True
AUDIO_DIR exists: True


In [7]:
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

# kalau error module not found, install dari notebook:
# !python -m pip install -U transformers accelerate peft datasets soundfile pandas numpy scikit-learn tqdm

from transformers import AutoFeatureExtractor, WavLMModel

MODEL_NAME = "microsoft/wavlm-base-plus"
feat = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
backbone = WavLMModel.from_pretrained(MODEL_NAME).to("cuda").eval()

print("Loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))


torch: 2.9.1+cu130 cuda: True


/venv/main/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loaded: microsoft/wavlm-base-plus
GPU: NVIDIA RTX A6000


In [9]:
import pandas as pd
from pathlib import Path

MANIFEST = Path("/workspace/ta_finetune/data/manifest_strict.csv")
AUDIO_DIR = Path("/workspace/ta_finetune/data/preprocessed_full")

df = pd.read_csv(MANIFEST)
print("rows:", len(df))
print("cols:", df.columns.tolist())
print(df["split_strict"].value_counts(dropna=False))

# cek kolom label Big5
traits = ["extraversion","neuroticism","agreeableness","conscientiousness","openness"]
print("missing traits:", [t for t in traits if t not in df.columns])

# cek beberapa file audio exist
def guess_path(row):
    # biasanya kamu punya clip_id atau file_name
    # sesuaikan salah satu:
    if "clip_id" in row:
        return AUDIO_DIR / f"{row['clip_id']}.wav"
    if "video_id" in row:
        return AUDIO_DIR / f"{row['video_id']}.wav"
    if "path" in row:
        return Path(row["path"])
    return None

sample = df.sample(10, random_state=42)
paths = [guess_path(r) for _, r in sample.iterrows()]
print("example paths:")
for p in paths[:5]:
    print(p, p.exists() if p else None)


rows: 9974
cols: ['clip_id', 'group_id', 'audio_out', 'gender', 'ethnicity', 'age_group', 'avg_trait', 'extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness', 'split_official', 'split_strict']
split_strict
train    5936
test     2039
val      1999
Name: count, dtype: int64
missing traits: []
example paths:
/workspace/ta_finetune/data/preprocessed_full/iW1t-ZiG2rc.000.wav False
/workspace/ta_finetune/data/preprocessed_full/WT1YjeADatU.001.wav False
/workspace/ta_finetune/data/preprocessed_full/pZxqWp0e-Ik.003.wav False
/workspace/ta_finetune/data/preprocessed_full/2c42A4Z7qPE.001.wav False
/workspace/ta_finetune/data/preprocessed_full/DWVJXufR7gE.001.wav False


In [17]:
import pandas as pd
from pathlib import Path

DATA = Path("/workspace/ta_finetune/data")
MANIFEST = DATA / "manifest_strict.csv"

df = pd.read_csv(MANIFEST)

AUDIO_ROOT = Path("/workspace/ta_finetune/data/preprocessed_full/preprocessed_full")

# tambahin kolom baru (absolute path)
df["audio_out_vast"] = df["audio_out"].apply(lambda s: str(AUDIO_ROOT / Path(s).name))

# health check
exists = df["audio_out_vast"].apply(lambda s: Path(s).exists())
print("✅ Exists:", int(exists.sum()), "/", len(exists))
print("❌ Missing:", int((~exists).sum()), "/", len(exists))

print("\nMissing per split:")
print(df.groupby("split_strict").apply(lambda g: int((~g["audio_out_vast"].apply(lambda s: Path(s).exists())).sum())))

# contoh missing
if (~exists).sum() > 0:
    display(df.loc[~exists, ["clip_id","audio_out","audio_out_vast","split_strict"]].head(15))

# simpan balik ke file yang sama (overwrite) atau simpan copy aman
# opsi aman: simpan file baru dengan kolom tambahan
OUT = DATA / "manifest_strict_with_vast.csv"
df.to_csv(OUT, index=False)
print("\nSaved:", OUT)


✅ Exists: 9974 / 9974
❌ Missing: 0 / 9974

Missing per split:
split_strict
test     0
train    0
val      0
dtype: int64

Saved: /workspace/ta_finetune/data/manifest_strict_with_vast.csv


/tmp/ipykernel_3089/905028082.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby("split_strict").apply(lambda g: int((~g["audio_out_vast"].apply(lambda s: Path(s).exists())).sum())))


In [18]:
import os, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ====== PATHS
ROOT = Path("/workspace/ta_finetune")
DATA = ROOT / "data"
OUT_ROOT = ROOT / "outputs" / "finetune_strict_wavlm"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST = DATA / "manifest_strict_with_vast.csv"  # yang sudah ada audio_out_vast
assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"

# ====== SEED
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| GPU:", torch.cuda.get_device_name(0))

# ====== TRAINING DEFAULT (sesuai diagram kamu)
MAX_EPOCHS = 20
PATIENCE = 5
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01

BATCH_SIZE = 16          # A6000 harusnya aman (kalau mau lebih cepat, coba 24/32)
NUM_WORKERS = 4
USE_AMP = True           # mixed precision

# ====== METRIC
TRAITS = ["extraversion","neuroticism","agreeableness","conscientiousness","openness"]

df = pd.read_csv(MANIFEST)
print("rows:", len(df))
print(df["split_strict"].value_counts())
print("cols ok:", all(t in df.columns for t in TRAITS))
print("audio_out_vast exists col:", "audio_out_vast" in df.columns)

# strict split
df_train = df[df["split_strict"]=="train"].copy()
df_val   = df[df["split_strict"]=="val"].copy()
df_test  = df[df["split_strict"]=="test"].copy()

print("train/val/test:", len(df_train), len(df_val), len(df_test))


DEVICE: cuda | GPU: NVIDIA RTX A6000
rows: 9974
split_strict
train    5936
test     2039
val      1999
Name: count, dtype: int64
cols ok: True
audio_out_vast exists col: True
train/val/test: 5936 1999 2039


In [19]:
import soundfile as sf
from transformers import AutoFeatureExtractor

MODEL_NAME = "microsoft/wavlm-base-plus"
feat = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

class StrictAudioDataset(Dataset):
    def __init__(self, df, is_train: bool):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = Path(row["audio_out_vast"])
        wav, sr = sf.read(p)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav = torch.tensor(wav, dtype=torch.float32)
        y = torch.tensor([row[t] for t in TRAITS], dtype=torch.float32)
        return {"wav": wav, "sr": sr, "y": y}

def collate_fn(batch):
    wavs = [b["wav"].numpy() for b in batch]
    srs = [b["sr"] for b in batch]
    assert len(set(srs)) == 1, f"sample rate beda2: {set(srs)}"
    inputs = feat(wavs, sampling_rate=srs[0], return_tensors="pt", padding=True)
    y = torch.stack([b["y"] for b in batch], dim=0)
    return inputs, y

train_ds = StrictAudioDataset(df_train, is_train=True)
val_ds   = StrictAudioDataset(df_val,   is_train=False)
test_ds  = StrictAudioDataset(df_test,  is_train=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True, collate_fn=collate_fn)

val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True,
                    persistent_workers=True, collate_fn=collate_fn)

test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True,
                     persistent_workers=True, collate_fn=collate_fn)

# quick sanity batch
inputs, y = next(iter(train_dl))
print({k:v.shape for k,v in inputs.items()}, y.shape)


{'input_values': torch.Size([16, 240000]), 'attention_mask': torch.Size([16, 240000])} torch.Size([16, 5])


In [35]:
from transformers import WavLMModel
from peft import LoraConfig, TaskType, get_peft_model

class WavLMWithHead(nn.Module):
    def __init__(self, backbone_name: str, r: int, lora_alpha: int = 32, lora_dropout: float = 0.05):
        super().__init__()
        base = WavLMModel.from_pretrained(backbone_name)

        # freeze backbone first (sesuai diagram)
        for p in base.parameters():
            p.requires_grad = False

        # LoRA on q_proj & v_proj
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["q_proj", "v_proj"],
            bias="none",
        )
        self.backbone = get_peft_model(base, lora_cfg)

        # simple regression head: mean-pool -> Linear(5)
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden, 5),
            nn.Sigmoid(),  # output 0..1 biar match label
        )

    def mean_pool(self, x, attn_mask=None):
        # x: (B,T,H), mask: (B,T)
        if attn_mask is None:
            return x.mean(dim=1)
        m = attn_mask.unsqueeze(-1).type_as(x)
        return (x * m).sum(dim=1) / (m.sum(dim=1).clamp(min=1.0))

    def forward(self, input_values, attention_mask=None):
        out = self.backbone(input_values=input_values, attention_mask=attention_mask)
        h = out.last_hidden_state  # (B, T_feat, H)
    
        feat_mask = None
        if attention_mask is not None:
            # ambil base model (karena backbone kamu sudah di-wrap PEFT)
            base = self.backbone.base_model if hasattr(self.backbone, "base_model") else self.backbone
            feat_mask = base._get_feature_vector_attention_mask(h.shape[1], attention_mask)
    
        pooled = self.mean_pool(h, feat_mask)
        return self.head(pooled)


def count_trainable(model):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, trn

# sanity build
m = WavLMWithHead(MODEL_NAME, r=8).to(DEVICE)
tot, trn = count_trainable(m)
print("params total:", tot, "| trainable:", trn)


params total: 94680693 | trainable: 298757


In [36]:
from torch.amp import autocast, GradScaler

import numpy as np
import torch

def eval_metrics(model, dl):
    model.eval()
    ys, yhats = [], []
    with torch.no_grad():
        for inputs, y in dl:
            iv = inputs["input_values"].to(DEVICE, non_blocking=True)
            am = inputs.get("attention_mask", None)
            if am is not None:
                am = am.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            yhat = model(iv, am)

            ys.append(y.detach().cpu())
            yhats.append(yhat.detach().cpu())

    y_true = torch.cat(ys, dim=0).numpy()    # (N,5)
    y_pred = torch.cat(yhats, dim=0).numpy() # (N,5)

    # MAE
    mae_per = np.mean(np.abs(y_pred - y_true), axis=0)     # (5,)
    mae_mean = float(np.mean(mae_per))

    # RMSE
    rmse_per = np.sqrt(np.mean((y_pred - y_true)**2, axis=0))
    rmse_mean = float(np.mean(rmse_per))

    # R2 per trait: 1 - SSE/SST
    # handle edge case: SST=0
    r2_per = []
    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        sse = np.sum((yt - yp)**2)
        sst = np.sum((yt - np.mean(yt))**2)
        r2 = 1.0 - (sse / sst) if sst > 1e-12 else np.nan
        r2_per.append(r2)
    r2_per = np.array(r2_per, dtype=float)
    r2_mean = float(np.nanmean(r2_per))

    # ChaLearn-style accuracy (sering ditulis Acc = 1 - MAE)
    acc_per = 1.0 - mae_per
    acc_mean = float(np.mean(acc_per))

    # skor seleksi kamu
    S = 1.0 - mae_mean  # = acc_mean (kalau definisinya mean per trait sama)

    return {
        "mae_per": mae_per, "mae_mean": mae_mean,
        "rmse_per": rmse_per, "rmse_mean": rmse_mean,
        "r2_per": r2_per, "r2_mean": r2_mean,
        "acc_per": acc_per, "acc_mean": acc_mean,
        "S": S,
    }


def train_run(run_name: str, lr: float, r: int, seed: int = 42):
    # seed per run
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    out_dir = OUT_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    model = WavLMWithHead(MODEL_NAME, r=r, lora_alpha=32, lora_dropout=0.05).to(DEVICE)

    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    scaler = GradScaler(enabled=USE_AMP)

    best_S = -1e9
    best_epoch = 0
    patience_cnt = 0

    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        pbar = tqdm(train_dl, desc=f"{run_name} | epoch {epoch}", leave=False)

        for inputs, y in pbar:
            iv = inputs["input_values"].to(DEVICE, non_blocking=True)
            am = inputs.get("attention_mask", None)
            if am is not None:
                am = am.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with autocast(device_type="cuda", enabled=USE_AMP):
                yhat = model(iv, am)
                loss = F.l1_loss(yhat, y)  # MAE mean all dims

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()

            pbar.set_postfix(loss=float(loss.item()))

        # ===== VALIDATION METRICS (end of epoch)
        m = eval_metrics(model, val_dl)

        mae_per, mae_mean, S = m["mae_per"], m["mae_mean"], m["S"]
        rmse_per, rmse_mean  = m["rmse_per"], m["rmse_mean"]
        r2_per, r2_mean      = m["r2_per"], m["r2_mean"]
        acc_per, acc_mean    = m["acc_per"], m["acc_mean"]

        rec = {
            "epoch": epoch,
            "lr": lr,
            "r": r,

            "mae_mean": mae_mean,
            "rmse_mean": rmse_mean,
            "r2_mean": r2_mean,
            "acc_mean": acc_mean,
            "S": S,

            "mae_extraversion": float(mae_per[0]),
            "mae_neuroticism": float(mae_per[1]),
            "mae_agreeableness": float(mae_per[2]),
            "mae_conscientiousness": float(mae_per[3]),
            "mae_openness": float(mae_per[4]),

            "rmse_extraversion": float(rmse_per[0]),
            "rmse_neuroticism": float(rmse_per[1]),
            "rmse_agreeableness": float(rmse_per[2]),
            "rmse_conscientiousness": float(rmse_per[3]),
            "rmse_openness": float(rmse_per[4]),

            "r2_extraversion": float(r2_per[0]),
            "r2_neuroticism": float(r2_per[1]),
            "r2_agreeableness": float(r2_per[2]),
            "r2_conscientiousness": float(r2_per[3]),
            "r2_openness": float(r2_per[4]),

            "acc_extraversion": float(acc_per[0]),
            "acc_neuroticism": float(acc_per[1]),
            "acc_agreeableness": float(acc_per[2]),
            "acc_conscientiousness": float(acc_per[3]),
            "acc_openness": float(acc_per[4]),
        }

        history.append(rec)

        print(
            f"[{run_name}] epoch={epoch} | "
            f"MAE_mean={mae_mean:.6f} | RMSE_mean={rmse_mean:.6f} | "
            f"R2_mean={r2_mean:.6f} | Acc_mean={acc_mean:.6f} | S={S:.6f}"
        )



        # early stopping on best S
        if S > best_S:
            best_S = S
            best_epoch = epoch
            patience_cnt = 0

            # save best checkpoint
            ckpt_dir = out_dir / "best"
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            model.backbone.save_pretrained(ckpt_dir)          # peft adapter + base ref
            torch.save(model.head.state_dict(), ckpt_dir / "head.pt")

            with open(ckpt_dir / "config.json", "w") as f:
                json.dump({
                    "run_name": run_name,
                    "model": MODEL_NAME,
                    "split": "strict",
                    "lr": lr,
                    "r": r,
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "best_S": best_S,
                    "metric": "S = 1 - MAE_mean",
                    "max_epochs": MAX_EPOCHS,
                    "patience": PATIENCE
                }, f, indent=2)

        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"[{run_name}] Early stopping (patience {PATIENCE})")
                break

    # save history csv
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(out_dir / "history.csv", index=False)

    return {"run_name": run_name, "best_S": best_S, "best_epoch": best_epoch, "out_dir": str(out_dir)}

print("done")


done


In [30]:
train_dl_dbg = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=0,            # <-- penting
    pin_memory=False,
    persistent_workers=False,
    collate_fn=collate_fn,
)

inputs, y = next(iter(train_dl_dbg))
print("OK batch:", {k:v.shape for k,v in inputs.items()}, y.shape)


OK batch: {'input_values': torch.Size([4, 240000]), 'attention_mask': torch.Size([4, 240000])} torch.Size([4, 5])


In [31]:
import soundfile as sf
from pathlib import Path

row = df_train.sample(1, random_state=42).iloc[0]
p = Path(row["audio_out_vast"])
print("path:", p, "exists:", p.exists())

info = sf.info(p)
print("sr:", info.samplerate, "dur:", info.duration, "frames:", info.frames, "channels:", info.channels)


path: /workspace/ta_finetune/data/preprocessed_full/preprocessed_full/LGdzP-r_G3U.004.wav exists: True
sr: 16000 dur: 15.0 frames: 240000 channels: 1


In [33]:
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True,
                      persistent_workers=False, collate_fn=collate_fn)

val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True,
                    persistent_workers=False, collate_fn=collate_fn)


In [38]:
# === QUICK SANITY: 1 train step + 1 val metrics
model = WavLMWithHead(MODEL_NAME, r=8, lora_alpha=32, lora_dropout=0.05).to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=WEIGHT_DECAY)
scaler = GradScaler(enabled=USE_AMP)

model.train()
inputs, y = next(iter(train_dl))
iv = inputs["input_values"].to(DEVICE, non_blocking=True)
am = inputs.get("attention_mask", None)
if am is not None:
    am = am.to(DEVICE, non_blocking=True)
y = y.to(DEVICE, non_blocking=True)

opt.zero_grad(set_to_none=True)
with autocast(device_type="cuda", enabled=USE_AMP):
    yhat = model(iv, am)
    loss = F.l1_loss(yhat, y)
scaler.scale(loss).backward()
scaler.unscale_(opt)
torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
scaler.step(opt)
scaler.update()

print("✅ one-step train loss:", float(loss.item()))

m = eval_metrics(model, val_dl)
print("✅ val metrics sample:",
      "MAE_mean", m["mae_mean"],
      "| RMSE_mean", m["rmse_mean"],
      "| R2_mean", m["r2_mean"],
      "| Acc_mean", m["acc_mean"])


TypeError: WavLMModel.forward() got an unexpected keyword argument 'input_ids'